# Intrusion Detection System — Data Preprocessing & Cleaning
### Dataset: CICIDS-2017

This notebook takes the **raw** CICIDS-2017 data and produces a **clean, model-ready** dataset.
It runs *after* `EDA.ipynb` and *before* feature engineering / modelling.

**Issues found in EDA that this notebook fixes:**
| # | Issue | Action |
|---|-------|--------|
| 1 | `Web Attack � ...` garbled label encoding | Fix label text |
| 2 | Inf values in `Flow Bytes/s`, `Flow Packets/s` | Replace Inf → NaN |
| 3 | NaN values in `Flow Bytes/s` (+ new NaNs from step 2) | Impute (median, per-class) |
| 4 | 256,479 duplicate rows (9.06%) | Drop duplicates |
| 5 | 8 zero-variance (constant) columns | Drop columns |
| 6 | 39 highly-correlated pairs (\|r\|>0.95) | Drop one feature per redundant pair |
| 7 | Negative values; `Init_Win` `-1` sentinel | Clip ordinary negatives; flag `Init_Win` sentinel, keep `-1` |
| 7b | 15 classes, several unlearnable (Heartbleed=11, etc.) | Group into 7 attack families |
| 8 | No train/test split | Stratified split (binary + multiclass labels) |

**Outputs (saved to Google Drive `IDS_Project/02_preprocessing/`):**
- `cicids2017_clean.parquet` — full cleaned dataset
- `train.parquet` / `test.parquet` — stratified split (already include `has_init_win_*` flags)
- `label_mapping.csv` — multi-class integer ↔ family-name mapping
- `Preprocessing_Report.txt` — every action logged
- `figures/` — before/after visual for every cleaning step

> **Note:** because `Init_Win` is handled here, the Feature Engineering notebook needs **no access to the raw dataset** — it only consumes `train.parquet` / `test.parquet`.

## 1. Imports, Config & Report Helpers

In [ ]:
import os, zipfile, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

LABEL_COL   = 'Label'
RANDOM_SEED = 42
TEST_SIZE   = 0.20

# ── persist all outputs to Google Drive so they survive runtime restarts ──
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/IDS_Project'   # central project folder on Drive
OUT_DIR     = os.path.join(PROJECT_DIR, '02_preprocessing')
FIGURES_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── report helpers ─────────────────────────────────────────────────
_report_lines = []

def _log(text=''):
    _report_lines.append(str(text))
    print(text)

def _savefig(name, fig=None):
    path = os.path.join(FIGURES_DIR, name)
    (fig or plt).savefig(path, dpi=130, bbox_inches='tight')
    return path

def write_report():
    path = os.path.join(OUT_DIR, 'Preprocessing_Report.txt')
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_report_lines))
    print(f'\nReport saved -> {path}')

_log('=' * 70)
_log('PREPROCESSING REPORT  —  CICIDS-2017')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
_log('=' * 70)
print('\nSetup complete.')
print('  Project dir          :', PROJECT_DIR)
print('  Preprocessing outputs:', OUT_DIR)

## 2. Load Raw Data
Same loading logic as `EDA.ipynb` — mount Drive, extract the zip, concatenate all 8 CSVs, strip column whitespace.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/MachineLearningCSV.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError(f'File not found: {zip_path}')

extract_dir = '/content/cicids2017'
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

dfs = []
for root, _, fnames in os.walk(extract_dir):
    for fname in sorted(fnames):
        if not fname.endswith('.csv'):
            continue
        df_tmp = pd.read_csv(os.path.join(root, fname), low_memory=False)
        df_tmp.columns = df_tmp.columns.str.strip()
        df_tmp['source_file'] = fname.split('.')[0]
        dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)
raw_shape = df.shape

_log('')
_log('── STEP 0 : RAW DATA LOADED ────────────────────────────────')
_log(f'  Raw shape : {raw_shape[0]:,} rows x {raw_shape[1]} cols')

## 3. Fix Garbled Label Encoding
The `Web Attack` labels contain a corrupted byte (`�`) from the original CSV encoding. Normalise them to clean text.

In [ ]:
labels_before = sorted(df[LABEL_COL].unique().tolist())

# Normalise: collapse any whitespace / corrupted bytes inside the label text
def clean_label(lbl):
    lbl = str(lbl).strip()
    # replace the corrupted separator in 'Web Attack � X' with a clean dash
    lbl = lbl.replace('\ufffd', '-').replace('\x96', '-')
    lbl = ' '.join(lbl.split())            # collapse multiple spaces
    lbl = lbl.replace(' - ', ' - ')
    return lbl

df[LABEL_COL] = df[LABEL_COL].apply(clean_label)
labels_after = sorted(df[LABEL_COL].unique().tolist())

_log('')
_log('── STEP 1 : LABEL ENCODING FIX ─────────────────────────────')
_log(f'  Labels before : {labels_before}')
_log(f'  Labels after  : {labels_after}')
print('\nCleaned label counts:')
print(df[LABEL_COL].value_counts())

## 4. Handle Infinite Values
EDA found Inf in `Flow Bytes/s` (1,509) and `Flow Packets/s` (2,867). These come from flows with near-zero duration. Replace Inf → NaN so they can be imputed in the next step.

In [ ]:
num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
feat_cols = [c for c in num_cols if c != 'source_file']

inf_before = {c: int(np.isinf(df[c]).sum()) for c in feat_cols if np.isinf(df[c]).any()}

# Replace +/- Inf with NaN across all numeric features
df[feat_cols] = df[feat_cols].replace([np.inf, -np.inf], np.nan)

inf_after = {c: int(np.isinf(df[c]).sum()) for c in feat_cols if np.isinf(df[c]).any()}

_log('')
_log('── STEP 2 : INFINITE VALUES ───────────────────────────────')
_log(f'  Inf values before : {inf_before}  (total {sum(inf_before.values()):,})')
_log(f'  Inf values after  : {inf_after if inf_after else "NONE"}')
_log('  -> all Inf replaced with NaN (imputed in next step)')

# before/after plot
fig, ax = plt.subplots(figsize=(8, 4))
cols   = list(inf_before.keys())
before = [inf_before[c] for c in cols]
after  = [inf_after.get(c, 0) for c in cols]
x = np.arange(len(cols))
ax.bar(x - 0.2, before, 0.4, label='Before', color='#F44336')
ax.bar(x + 0.2, after,  0.4, label='After',  color='#4CAF50')
ax.set_xticks(x); ax.set_xticklabels(cols, rotation=15)
ax.set_ylabel('Inf value count')
ax.set_title('Infinite Values — Before vs After')
for i, v in enumerate(before):
    ax.text(i - 0.2, v, f' {v:,}', ha='center', va='bottom', fontsize=8)
ax.legend()
plt.tight_layout()
_savefig('01_inf_before_after.png', fig)
plt.show()

## 5. Handle Missing (NaN) Values
After the Inf→NaN replacement, the affected columns now have NaNs. Impute with the **per-class median** (median of the same attack/benign class) — this avoids leaking benign statistics into attack rows.

In [ ]:
nan_before = df[feat_cols].isnull().sum()
nan_before = nan_before[nan_before > 0].sort_values(ascending=False)

# Per-class median imputation
for col in nan_before.index:
    df[col] = df.groupby(LABEL_COL)[col].transform(lambda s: s.fillna(s.median()))

# Any residual NaN (a whole class was NaN for that col) -> global median
residual = df[feat_cols].isnull().sum()
residual = residual[residual > 0]
for col in residual.index:
    df[col] = df[col].fillna(df[col].median())

nan_after = df[feat_cols].isnull().sum().sum()

_log('')
_log('── STEP 3 : MISSING VALUES ────────────────────────────────')
_log(f'  NaN before (per col):')
for c, n in nan_before.items():
    _log(f'    {c:<30} {n:>7,}')
_log(f'  Imputation strategy : per-class median (fallback: global median)')
_log(f'  NaN remaining after : {nan_after}')

# before/after plot
fig, ax = plt.subplots(figsize=(8, 4))
cols = list(nan_before.index)
x = np.arange(len(cols))
ax.bar(x - 0.2, nan_before.values, 0.4, label='Before', color='#FF9800')
ax.bar(x + 0.2, [0] * len(cols),   0.4, label='After',  color='#4CAF50')
ax.set_xticks(x); ax.set_xticklabels(cols, rotation=15)
ax.set_ylabel('NaN count')
ax.set_title('Missing Values — Before vs After')
for i, v in enumerate(nan_before.values):
    ax.text(i - 0.2, v, f' {v:,}', ha='center', va='bottom', fontsize=8)
ax.legend()
plt.tight_layout()
_savefig('02_nan_before_after.png', fig)
plt.show()

## 6. Remove Duplicate Rows
EDA found 256,479 exact duplicate rows (9.06%). Duplicates leak between train/test and bias statistics — drop them, keeping the first occurrence.

In [ ]:
rows_before   = len(df)
labels_b4_dup = df[LABEL_COL].value_counts()

df = df.drop_duplicates().reset_index(drop=True)

rows_after    = len(df)
n_removed     = rows_before - rows_after
labels_post   = df[LABEL_COL].value_counts()

_log('')
_log('── STEP 4 : DUPLICATE ROWS ────────────────────────────────')
_log(f'  Rows before : {rows_before:,}')
_log(f'  Rows after  : {rows_after:,}')
_log(f'  Removed     : {n_removed:,}  ({n_removed/rows_before*100:.2f}%)')

# before/after plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(['Before', 'After'], [rows_before, rows_after],
            color=['#F44336', '#4CAF50'])
axes[0].set_title(f'Total Rows  (removed {n_removed:,})')
axes[0].set_ylabel('Row count')
for i, v in enumerate([rows_before, rows_after]):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')

comp = pd.DataFrame({'Before': labels_b4_dup, 'After': labels_post}).fillna(0)
comp.plot(kind='bar', ax=axes[1], color=['#F44336', '#4CAF50'])
axes[1].set_title('Rows per Label — Before vs After Dedup')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=40)
axes[1].set_yscale('log')
plt.tight_layout()
_savefig('03_duplicates_before_after.png', fig)
plt.show()

## 7. Drop Zero-Variance Columns
8 columns are constant (single value for every row). They carry zero information — drop them. Recomputed here rather than hard-coded, so the notebook stays correct if the data changes.

In [ ]:
num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
feat_cols = [c for c in num_cols if c != 'source_file']

variances     = df[feat_cols].var()
zero_var_cols = variances[variances == 0].index.tolist()

cols_before = df.shape[1]
df = df.drop(columns=zero_var_cols)
cols_after  = df.shape[1]

_log('')
_log('── STEP 5 : ZERO-VARIANCE COLUMNS ──────────────────────────')
_log(f'  Zero-variance columns dropped ({len(zero_var_cols)}):')
for c in zero_var_cols:
    _log(f'    - {c}')
_log(f'  Columns: {cols_before} -> {cols_after}')

# plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(zero_var_cols)), [1] * len(zero_var_cols), color='#9E9E9E')
ax.set_xticks(range(len(zero_var_cols)))
ax.set_xticklabels(zero_var_cols, rotation=45, ha='right', fontsize=8)
ax.set_title(f'Dropped Zero-Variance Columns ({len(zero_var_cols)})')
ax.set_ylabel('Variance (= 0)')
ax.set_ylim(0, 2)
plt.tight_layout()
_savefig('04_zero_variance_dropped.png', fig)
plt.show()

## 8. Drop Highly-Correlated Redundant Features
EDA found 39 feature pairs with \|r\| > 0.95. For each redundant pair we keep one feature and drop the other. We greedily drop the feature that appears in the most high-correlation pairs first.

In [ ]:
CORR_THRESHOLD = 0.95

num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
feat_cols = [c for c in num_cols if c not in ('source_file', 'label_binary', 'label_multi')]

corr  = df[feat_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

# all pairs above threshold
pairs = (upper.stack()
              .reset_index()
              .rename(columns={'level_0': 'A', 'level_1': 'B', 0: 'r'}))
pairs = pairs[pairs['r'] > CORR_THRESHOLD].sort_values('r', ascending=False)

# greedy drop: repeatedly drop the feature involved in the most remaining pairs
to_drop   = set()
remaining = pairs.copy()
while not remaining.empty:
    counts = pd.concat([remaining['A'], remaining['B']]).value_counts()
    victim = counts.index[0]
    to_drop.add(victim)
    remaining = remaining[(remaining['A'] != victim) & (remaining['B'] != victim)]

to_drop    = sorted(to_drop)
kept_feats = [c for c in feat_cols if c not in to_drop]

cols_before = df.shape[1]
df = df.drop(columns=to_drop)
cols_after  = df.shape[1]

_log('')
_log('── STEP 6 : HIGHLY-CORRELATED FEATURES ─────────────────────')
_log(f'  Correlation threshold : |r| > {CORR_THRESHOLD}')
_log(f'  High-correlation pairs found : {len(pairs)}')
_log(f'  Features dropped ({len(to_drop)}):')
for c in to_drop:
    _log(f'    - {c}')
_log(f'  Columns: {cols_before} -> {cols_after}')

# before/after correlation heatmaps
corr_after = corr.loc[kept_feats, kept_feats]
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
sns.heatmap(corr, ax=axes[0], cmap='coolwarm', center=0, vmin=0, vmax=1,
            xticklabels=False, yticklabels=False, cbar=True)
axes[0].set_title(f'Before — {len(feat_cols)} features')
sns.heatmap(corr_after, ax=axes[1], cmap='coolwarm', center=0, vmin=0, vmax=1,
            xticklabels=False, yticklabels=False, cbar=True)
axes[1].set_title(f'After — {len(kept_feats)} features (redundancy removed)')
plt.suptitle('Correlation Structure — Before vs After', fontsize=13)
plt.tight_layout()
_savefig('05_correlation_before_after.png', fig)
plt.show()

## 9. Handle Negative Values (with `Init_Win` sentinel awareness)
Several flow-statistic columns are physically non-negative (counts, durations, lengths) — a negative there is a measurement artefact, so we clip to 0.

**Exception — `Init_Win_bytes_forward` / `Init_Win_bytes_backward`:** in CICIDS-2017 a value of `-1` is a *sentinel* meaning "no TCP window value was observed" (e.g. non-TCP flows). Blindly clipping `-1 → 0` would make "no window observed" indistinguishable from "window size 0". So for those two columns we instead:
- add `has_init_win_fwd` / `has_init_win_bwd` binary flags (1 = real value observed, 0 = sentinel)
- keep the `-1` as `-1` so the model sees it as its own category

This way the sentinel signal is preserved and feature engineering needs **no access to the raw data**.

In [ ]:
num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
feat_cols = [c for c in num_cols if c != 'source_file']

INIT_COLS = ['Init_Win_bytes_forward', 'Init_Win_bytes_backward']

_log('')
_log('── STEP 7 : NEGATIVE VALUES (Init_Win sentinel-aware) ──────')

# ── 7a. Init_Win sentinel handling — add flags, keep -1 as its own value ──
for col, flag in zip(INIT_COLS, ['has_init_win_fwd', 'has_init_win_bwd']):
    n_sentinel = int((df[col] < 0).sum())
    df[flag] = (df[col] >= 0).astype(int)        # 1 = real value observed, 0 = sentinel
    # clip any value below -1 up to -1 (only -1 is a valid sentinel)
    df[col] = df[col].clip(lower=-1)
    _log(f'  {col:<26} sentinel(-1) rows : {n_sentinel:>9,}  -> flag {flag} added')

# ── 7b. ordinary negative clipping for every other feature ──
other_cols = [c for c in feat_cols if c not in INIT_COLS]
neg_counts = {c: int((df[c] < 0).sum()) for c in other_cols if (df[c] < 0).any()}

if not neg_counts:
    _log('  Other columns : no negative values found.')
else:
    _log(f'  Other columns with negatives ({len(neg_counts)}) — clipped to 0:')
    for c, n in sorted(neg_counts.items(), key=lambda kv: -kv[1]):
        _log(f'    {c:<35} {n:>7,}')
    for c in neg_counts:
        df[c] = df[c].clip(lower=0)

# refresh feature list with the 2 new flag columns
feat_cols = feat_cols + ['has_init_win_fwd', 'has_init_win_bwd']

# ── plots ──
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# left: ordinary negative clipping
if neg_counts:
    cols = list(neg_counts.keys())
    axes[0].barh(cols, [neg_counts[c] for c in cols], color='#FF7043', label='Before')
    axes[0].barh(cols, [0] * len(cols), color='#4CAF50', label='After')
    axes[0].set_xlabel('Negative value count')
    axes[0].set_title('Ordinary Negatives — Before vs After Clipping')
    axes[0].legend()
else:
    axes[0].text(0.5, 0.5, 'No ordinary negatives found', ha='center', va='center',
                 fontsize=13, color='green', transform=axes[0].transAxes)
    axes[0].set_axis_off()

# right: Init_Win sentinel fraction per attack family
sent = pd.DataFrame({
    'fwd': df.groupby(LABEL_COL)['has_init_win_fwd'].mean(),
    'bwd': df.groupby(LABEL_COL)['has_init_win_bwd'].mean(),
}).sort_values('fwd')
sent.plot(kind='barh', ax=axes[1], color=['#1976D2', '#FFA000'])
axes[1].set_title('Init_Win — fraction with REAL value (vs -1 sentinel) per label')
axes[1].set_xlabel('fraction observed')
axes[1].set_xlim(0, 1)

plt.suptitle('Negative Value Handling', fontsize=13)
plt.tight_layout()
_savefig('06_negatives.png', fig)
plt.show()

## 9b. Group Labels into Attack Families
The raw data has **15 classes**, but the bottom ones are unlearnable after a train/test split (Heartbleed = 11 rows, Sql Injection = 21, Infiltration = 36). Instead of a meaningless *"Other"* bucket, we group by **attack family** — semantically related attacks that share similar flow behaviour. This yields **7 well-populated, learnable classes**.

| Grouped class | Original labels |
|---|---|
| BENIGN | BENIGN |
| DoS | DoS Hulk, DoS GoldenEye, DoS slowloris, DoS Slowhttptest, Heartbleed |
| DDoS | DDoS |
| PortScan | PortScan |
| Brute Force | FTP-Patator, SSH-Patator |
| Web Attack | Web Attack Brute Force, Web Attack XSS, Web Attack Sql Injection |
| Bot/Infiltration | Bot, Infiltration |

The original 15-label column is kept as `Label` for reference; the grouped column becomes `Label_grouped`.

In [ ]:
# Map every raw label to its attack family.
# Keys are matched after clean_label() ran in Step 1, so 'Web Attack - XSS' etc.
FAMILY_MAP = {
    'BENIGN':                     'BENIGN',
    'DoS Hulk':                   'DoS',
    'DoS GoldenEye':              'DoS',
    'DoS slowloris':              'DoS',
    'DoS Slowhttptest':           'DoS',
    'Heartbleed':                 'DoS',
    'DDoS':                       'DDoS',
    'PortScan':                   'PortScan',
    'FTP-Patator':                'Brute Force',
    'SSH-Patator':                'Brute Force',
    'Web Attack - Brute Force':   'Web Attack',
    'Web Attack - XSS':           'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
    'Bot':                        'Bot/Infiltration',
    'Infiltration':               'Bot/Infiltration',
}

# Safety check: every label in the data must have a mapping
unmapped = set(df[LABEL_COL].unique()) - set(FAMILY_MAP)
if unmapped:
    raise ValueError(f'Unmapped labels (add them to FAMILY_MAP): {unmapped}')

counts_15 = df[LABEL_COL].value_counts()
df['Label_grouped'] = df[LABEL_COL].map(FAMILY_MAP)
counts_7 = df['Label_grouped'].value_counts()

_log('')
_log('── STEP 7b : LABEL GROUPING (15 -> 7 families) ─────────────')
_log(f'  Original classes : {len(counts_15)}')
_log(f'  Grouped classes  : {len(counts_7)}')
_log('  Grouped distribution:')
for fam, n in counts_7.items():
    _log(f'    {fam:<20} {n:>10,}  ({n/len(df)*100:.2f}%)')

print('\nGrouped class distribution:')
print(counts_7)

# before/after plot
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

counts_15.plot(kind='bar', ax=axes[0],
               color=sns.color_palette('tab20', len(counts_15)), edgecolor='white', logy=True)
axes[0].set_title(f'Before — {len(counts_15)} raw classes (log scale)')
axes[0].set_ylabel('Count (log)')
axes[0].tick_params(axis='x', rotation=45)

bar_colors = sns.color_palette('tab10', len(counts_7))
counts_7.plot(kind='bar', ax=axes[1], color=bar_colors, edgecolor='white')
axes[1].set_title(f'After — {len(counts_7)} attack families')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=8)

plt.suptitle('Label Grouping — 15 raw classes -> 7 attack families', fontsize=13)
plt.tight_layout()
_savefig('07b_label_grouping.png', fig)
plt.show()

## 10. Create Label Columns (Binary & Multi-class)
Add two model-ready target columns:
- `label_binary` : 0 = BENIGN, 1 = ATTACK
- `label_multi`  : integer-encoded attack class (mapping saved to report)

In [ ]:
# Binary target (BENIGN vs ATTACK) — derived from the raw label
df['label_binary'] = (df[LABEL_COL] != 'BENIGN').astype(int)

# Multi-class target — integer-encoded from the GROUPED label (7 families)
classes   = sorted(df['Label_grouped'].unique())
multi_map = {name: i for i, name in enumerate(classes)}
df['label_multi'] = df['Label_grouped'].map(multi_map)

_log('')
_log('── STEP 8 : LABEL ENCODING ───────────────────────────────')
_log('  label_binary : 0 = BENIGN, 1 = ATTACK')
_log(f'    {dict(df["label_binary"].value_counts())}')
_log('  label_multi  : integer-encoded from Label_grouped (7 families)')
for name, idx in multi_map.items():
    _log(f'    {idx:>2} = {name}')

print('Binary distribution:')
print(df['label_binary'].value_counts())
print('\nMulti-class (grouped) distribution:')
print(df['label_multi'].value_counts().sort_index())

## 11. Stratified Train/Test Split
80/20 split, **stratified on the multi-class label** so even rare classes (Heartbleed, Infiltration) appear in both sets in proportion. The same split serves binary and multi-class tasks.

In [ ]:
# Drop the rarest classes' risk: stratify needs >=2 samples per class. Check first.
vc = df['label_multi'].value_counts()
too_rare = vc[vc < 2]
if not too_rare.empty:
    _log(f'  WARNING: classes with <2 samples cannot be stratified: {too_rare.to_dict()}')

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df['label_multi'],
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

_log('')
_log('── STEP 9 : TRAIN/TEST SPLIT ─────────────────────────────')
_log(f'  Split ratio : {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)}  (stratified on label_multi)')
_log(f'  Train rows  : {len(train_df):,}')
_log(f'  Test rows   : {len(test_df):,}')

# stratification check plot
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

prop = pd.DataFrame({
    'Train': train_df[LABEL_COL].value_counts(normalize=True),
    'Test':  test_df[LABEL_COL].value_counts(normalize=True),
}).fillna(0) * 100
prop.plot(kind='bar', ax=axes[0], color=['#1976D2', '#FFA000'])
axes[0].set_title('Class Proportions — Train vs Test (should match)')
axes[0].set_ylabel('% of split')
axes[0].tick_params(axis='x', rotation=40)

counts = pd.DataFrame({
    'Train': train_df[LABEL_COL].value_counts(),
    'Test':  test_df[LABEL_COL].value_counts(),
}).fillna(0)
counts.plot(kind='bar', ax=axes[1], color=['#1976D2', '#FFA000'], logy=True)
axes[1].set_title('Class Counts — Train vs Test (log scale)')
axes[1].set_ylabel('Count (log)')
axes[1].tick_params(axis='x', rotation=40)

plt.suptitle('Stratified Split Verification', fontsize=13)
plt.tight_layout()
_savefig('07_train_test_split.png', fig)
plt.show()

## 12. Save Cleaned Dataset & Splits

In [ ]:
clean_path = os.path.join(OUT_DIR, 'cicids2017_clean.parquet')
train_path = os.path.join(OUT_DIR, 'train.parquet')
test_path  = os.path.join(OUT_DIR, 'test.parquet')

df.to_parquet(clean_path, index=False)
train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)

# also save the multi-class label mapping as csv for downstream use
map_path = os.path.join(OUT_DIR, 'label_mapping.csv')
pd.Series(multi_map, name='index').rename_axis('label').to_csv(map_path)

_log('')
_log('── STEP 10 : FILES SAVED ────────────────────────────────')
_log(f'  {clean_path}')
_log(f'  {train_path}')
_log(f'  {test_path}')
_log(f'  {map_path}')

for p in [clean_path, train_path, test_path]:
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {os.path.basename(p):28} {size_mb:8.1f} MB')

## 13. Final Summary & Report

In [ ]:
_log('')
_log('=' * 70)
_log('SUMMARY  —  RAW vs CLEAN')
_log('=' * 70)
_log(f'  Rows    : {raw_shape[0]:>12,}  ->  {len(df):>12,}   ({len(df)-raw_shape[0]:+,})')
_log(f'  Columns : {raw_shape[1]:>12}  ->  {df.shape[1]:>12}   ({df.shape[1]-raw_shape[1]:+})')
_log(f'  Classes : {15:>12}  ->  {df["label_multi"].nunique():>12}   (grouped into attack families)')
_log('')
_log('  Cleaning actions applied:')
_log('    1. Fixed garbled Web Attack label encoding')
_log('    2. Replaced Inf -> NaN')
_log('    3. Imputed NaN with per-class median')
_log('    4. Dropped exact duplicate rows')
_log('    5. Dropped zero-variance columns')
_log('    6. Dropped highly-correlated redundant features')
_log('    7. Clipped ordinary negatives; flagged Init_Win sentinel (kept -1)')
_log('       -> added has_init_win_fwd, has_init_win_bwd')
_log('    7b. Grouped 15 raw classes -> 7 attack families')
_log('    8. Added label_binary + label_multi targets')
_log('    9. Stratified 80/20 train/test split')
_log('')
_log('  Output files: cicids2017_clean.parquet, train.parquet, test.parquet,')
_log('                label_mapping.csv, Preprocessing_Report.txt, figures/')
_log('')
_log('  Next step -> Feature Engineering (skew transforms, scaling, SMOTE on train)')
_log('               FE consumes train/test.parquet only — no raw data needed.')

# figure index
FIGURE_INDEX = [
    ('01_inf_before_after.png',          'Infinite values before/after'),
    ('02_nan_before_after.png',          'Missing values before/after'),
    ('03_duplicates_before_after.png',   'Duplicate rows before/after'),
    ('04_zero_variance_dropped.png',     'Zero-variance columns dropped'),
    ('05_correlation_before_after.png',  'Correlation structure before/after'),
    ('06_negatives.png',                 'Negative clipping + Init_Win sentinel flags'),
    ('07b_label_grouping.png',           '15 raw classes grouped into 7 families'),
    ('07_train_test_split.png',          'Stratified split verification'),
]
_log('')
_log('  Figures:')
for fname, desc in FIGURE_INDEX:
    _log(f'    {fname:<35} {desc}')

write_report()

print('\n' + '=' * 55)
print('PREPROCESSING COMPLETE')
print('=' * 55)
print(f'  Raw   : {raw_shape[0]:,} rows x {raw_shape[1]} cols, 15 classes')
print(f'  Clean : {len(df):,} rows x {df.shape[1]} cols, {df["label_multi"].nunique()} classes')
print(f'  Train : {len(train_df):,}  |  Test : {len(test_df):,}')
print(f'  Report  -> {OUT_DIR}/Preprocessing_Report.txt')
print(f'  Figures -> {FIGURES_DIR}/  ({len(FIGURE_INDEX)} figures)')